# Flat Valence Bands of Twisted Bilayer h-BN

> **Lede Xian, Dante M. Kennes, Nicolas Tancogne-Dejean, Massimo Altarelli, and Angel Rubio**
> **Multiflat Bands and Strong Correlations in Twisted Bilayer Boron Nitride: Doping-Induced Correlated Insulator and Superconductor**
> Nano Letters 19, 4934 (2019)
> [DOI: 10.1021/acs.nanolett.9b00986](https://doi.org/10.1021/acs.nanolett.9b00986)

Calculate the band structure of the twisted h-BN bilayers created in the
[structure notebook](interface_bilayer_twisted_commensurate_lattices_boron_nitride.ipynb), using Quantum ESPRESSO and the
band structure workflow from Standata, and read off the width of their top set of flat valence bands.

The article's result is that twisting bilayer h-BN splits its top valence bands into isolated flat sets whose width
shrinks with the twist angle, from 101.4 meV at 7.34° to 1.6 meV at 2.64° (Fig. 1g).

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](interface_bilayer_twisted_commensurate_lattices_boron_nitride.ipynb), which saves them to the `uploads` folder under the names used in cell 1.2 below.
1. Set the materials and the calculation parameters in cells 1.2 and 1.3 (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results; section 8 prints the comparison with the paper.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the materials, workflow, model, compute resources and jobs.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the materials by name from the `uploads` folder, print their provenance and save them to the platform.
1. Configure the workflow: select the application and the model, load the band structure workflow from Standata, and set the k-grid of the self-consistent run, the k-path of the bands and the cutoffs.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Run one job per material, one after another, re-using a job that already ran under the same workflow name.
1. Retrieve results: the band structure, the width of the top valence flat bands and their separation from the bands below, and the bands near the valence-band top.
1. Compare with Xian et al. (2019).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and jobs

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"
# Names as saved by the structure notebook, each with the width of its top valence flat set in Xian et al. 2019
# (meV, configuration not stated; 7.34° and 2.64°: text p. 4936; 6.01, 5.08 and 3.89°: read off Fig. 1g). One job per
# material; every angle below 7.34° exceeds one 4 h single-node job.
MATERIALS = {
    "h-BN bilayer 7.34deg alpha d3.23": {"paper_flat_band_width": 101.4},
    # "h-BN bilayer 7.34deg beta d3.23": {"paper_flat_band_width": 101.4},
    # "h-BN bilayer 6.01deg alpha d3.23": {"paper_flat_band_width": 54.9},
    # "h-BN bilayer 6.01deg beta d3.23": {"paper_flat_band_width": 54.9},
    # "h-BN bilayer 5.08deg alpha d3.23": {"paper_flat_band_width": 31.6},
    # "h-BN bilayer 5.08deg beta d3.23": {"paper_flat_band_width": 31.6},
    # "h-BN bilayer 3.89deg alpha d3.23": {"paper_flat_band_width": 11.7},
    # "h-BN bilayer 3.89deg beta d3.23": {"paper_flat_band_width": 11.7},
    # "h-BN bilayer 2.64deg alpha d3.23": {"paper_flat_band_width": 1.6},
    # "h-BN bilayer 2.64deg beta d3.23": {"paper_flat_band_width": 1.6},
}

WORKFLOW_SEARCH_TERM = "band_structure.json"
MY_WORKFLOW_NAME = "Band Structure"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 16  # OR allows 16 cores per job
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set the DFT parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV
ECUTWFC = 40  # Ry
ECUTRHO = 200  # Ry; paper: VASP PAW, 400 eV

KGRID = [1, 1, 1]  # Γ only, as the paper's self-consistent run
# paper: Γ–K–M, number of points not stated; ours 7, evenly spaced (|ΓK| = 2|KM|)
KPATH = [
    {"point": "Γ", "steps": 4},
    {"point": "K", "steps": 2},
    {"point": "M", "steps": 1},
]
MODEL_TAG = (f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KGRID[0]} "
             f"p{sum(point['steps'] for point in KPATH)}")

SCF_UNIT = "pw_scf"
BANDS_UNIT = "pw_bands"

PLOT_ENERGY_RANGE = (-0.5, 0.1)  # eV, relative to the top of the flat bands

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder and print provenance

The structures, their geometry and their lattice type all come from the structure notebook; this one only loads them
by name.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import load_material

materials = {}
for name in MATERIALS:
    material = load_material(client, FOLDER, name, ACCOUNT_ID)
    materials[name] = material
    print(f"{name}: {material.basis.number_of_atoms} atoms, a = {material.lattice.a:.3f} Å, "
          f"gamma = {material.lattice.gamma:.3f}°, lattice {material.lattice.type}")

### 3.2. Preview the materials

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

visualize_materials([{"material": material, "title": name} for name, material in materials.items()],
                    viewer="wave")

### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {}
for name, material in materials.items():
    material.basis.set_labels_from_list([])
    saved_materials[name] = Material.create(get_or_create_material(client, material, ACCOUNT_ID))

## 4. Configure the workflow
### 4.1. Select application and model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")

### 4.2. Configure the workflow

One workflow per material: the self-consistent run on `KGRID`, then the bands along `KPATH`, both with the cutoffs of
1.3. Each workflow is named after its material and `MODEL_TAG`; section 6 finds a job that already ran by that name.

The structure is not relaxed, as in the article's band-width series.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider, \
    PointsPathDataProvider

from copy import deepcopy

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
cutoffs_context = PlanewaveCutoffsContextProvider(
    wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).get_context_item_data()
path_context = PointsPathDataProvider(path=KPATH, isEdited=True).get_context_item_data()

workflows = {}
for name, material in materials.items():
    workflow = Workflow.create(deepcopy(workflow_config))
    workflow.name = f"{MY_WORKFLOW_NAME} {name} {MODEL_TAG}"
    subworkflow = workflow.subworkflows[0]
    subworkflow.model = model
    grid_context = PointsGridDataProvider(material=material, dimensions=KGRID, isEdited=True).get_context_item_data()

    for unit_name, contexts in [(SCF_UNIT, [grid_context, cutoffs_context]),
                                (BANDS_UNIT, [path_context, cutoffs_context])]:
        unit = subworkflow.get_unit_by_name(name=unit_name)
        for context in contexts:
            unit.add_context(context)
        subworkflow.set_unit(unit)
    workflows[name] = workflow
    print(workflow.name)

### 4.3. Preview the workflow

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

visualize_workflow(next(iter(workflows.values())))

### 4.4. Save the workflows to the collection

In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

for name, workflow in workflows.items():
    saved = Workflow.create(get_or_create_workflow(client, workflow, ACCOUNT_ID))
    print(f"{name}: workflow {saved.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Run the jobs one at a time

A job is found by its material and its workflow name, which carries the model tag, and re-used if it exists;
otherwise it is created and submitted. The jobs run one after another.

In [ ]:
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

job_ids = {}
for name, workflow in workflows.items():
    job = find_job_for_material(
        client, saved_materials[name].id, workflow.name, ACCOUNT_ID,
        statuses=("submitted", "queued", "active", "finished"),
    )
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_materials[name]], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ {name}: submitted job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing job {job['_id']}")
    await wait_for_jobs_to_finish_async(client.jobs, [job["_id"]], poll_interval=POLL_INTERVAL)
    job_ids[name] = job["_id"]

## 7. Retrieve the results
### 7.1. Band structure

The path is Γ–K–M of the moiré cell, as in the article; the bands of the two layers are folded onto its small
Brillouin zone.

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

band_structures = {}
for name, job_id in job_ids.items():
    band_structures[name] = get_properties_for_job(client, job_id, property_name="band_structure")[0]
    visualize_properties(band_structures[name], title=f"Band Structure: {name}",
                         extra_config={"material": materials[name].to_dict()})

### 7.2. Top valence flat bands

The flat set starts at the highest occupied band and takes in each band below whose maximum along the path reaches
the set's minimum. Its width is the spread of the set over the points of `KPATH`, and its separation is the distance
from the set's minimum down to the maximum of the next band. The article does not say whether its width is taken along
the path or over the whole Brillouin zone.

In [ ]:
import numpy as np

flat_bands = {}
for name, band_structure in band_structures.items():
    number_of_occupied_bands = 2 * materials[name].basis.number_of_atoms  # GBRV valence electrons: B 3, N 5
    bands = np.array(band_structure["yDataSeries"])  # band × path point, eV
    flat_set_bottom = number_of_occupied_bands - 1
    while bands[flat_set_bottom - 1].max() >= bands[flat_set_bottom:number_of_occupied_bands].min():
        flat_set_bottom -= 1
    flat_set = bands[flat_set_bottom:number_of_occupied_bands]
    flat_bands[name] = {
        "bands": bands,
        "flat_set": flat_set,
        "width": 1000 * (flat_set.max() - flat_set.min()),
        "separation": 1000 * (flat_set.min() - bands[flat_set_bottom - 1].max()),
    }

### 7.3. Bands near the valence-band top

Energies relative to the top of the flat set.

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

vertex_indices = np.cumsum([0] + [point["steps"] for point in KPATH[:-1]])
for name, result in flat_bands.items():
    figure, axes = plt.subplots()
    axes.plot((result["bands"] - result["flat_set"].max()).T, color="black", marker=".")
    axes.set_xticks(vertex_indices, [point["point"] for point in KPATH])
    axes.set_ylim(*PLOT_ENERGY_RANGE)
    axes.set_ylabel("E − E(top of the flat bands) (eV)")
    axes.set_title(f"Bands near the valence-band top: {name}")
    display_matplotlib_figure(figure)

## 8. Compare with Xian et al. (2019)

The article's width for each material, set in `MATERIALS`, beside ours; the article gives no separation.

Settings that differ from the article's: GBRV ultrasoft pseudopotentials at 40/200 Ry against VASP PAW at 400 eV, and
Standata's lattice constant of 2.509 Å against 2.512 Å.

In [ ]:
for name, result in flat_bands.items():
    print(f"{name}: top valence flat set ({len(result['flat_set'])} bands), width along Γ–K–M: "
          f"{result['width']:.1f} meV · Xian et al. 2019: {MATERIALS[name]['paper_flat_band_width']} meV")
    print(f"{name}: separation from the bands below: {result['separation']:.1f} meV")

## References

[1] L. Xian, D. M. Kennes, N. Tancogne-Dejean, M. Altarelli and A. Rubio, "Multiflat Bands and Strong Correlations in Twisted Bilayer Boron Nitride: Doping-Induced Correlated Insulator and Superconductor", Nano Lett. 19, 4934 (2019). https://doi.org/10.1021/acs.nanolett.9b00986

[2] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053